# VPD Stress Mechanisms and Cattle Mortality Models

This notebook examines physiological mechanisms linking VPD to cattle mortality and develops predictive models.

## Scientific Framework

### Physiological Mechanisms

**Vapor Pressure Deficit (VPD)** affects cattle through multiple pathways:

1. **Evaporative Cooling Impairment**
   - High VPD (dry air) initially enhances evaporative cooling
   - BUT: Depletes body water faster, risking dehydration
   - Respiratory water loss increases with VPD
   - Panting becomes less effective at extreme VPD

2. **Dehydration Cascade**
   - Water loss → reduced blood volume → cardiovascular strain
   - Reduced milk production in dairy cattle
   - Impaired thermoregulation due to water deficit

3. **Compound Heat Stress**
   - VPD × Temperature interaction
   - Hot, dry conditions maximize stress
   - Cumulative burden over days/weeks

4. **Feed and Water Intake**
   - High VPD suppresses appetite
   - Increased water requirements
   - Reduced digestive efficiency

### Heat Stress Indices

We'll examine several compound stress metrics:

- **Temperature-Humidity Index (THI)**: Traditional metric
- **VPD-Temperature Product**: Multiplicative stress
- **Cumulative VPD Exposure**: Time-integrated burden
- **VPD Threshold Exceedance**: Days above critical levels

## Research Objectives

1. Identify VPD thresholds associated with elevated mortality
2. Quantify compound stress (VPD + temperature) effects
3. Model dose-response relationships
4. Develop predictive models for mortality risk
5. Assess cumulative vs. acute exposure impacts
6. Identify high-risk periods and vulnerabilities

## Methods

- Threshold identification (piecewise regression, GAMs)
- Multiple regression with interaction terms
- Generalized Additive Models (GAMs)
- Time-weighted exposure metrics
- Cross-validation for model assessment
- Risk stratification analysis

In [ ]:
# Standard imports
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from scipy.optimize import curve_fit
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
# Note: GAM functionality removed due to compatibility issues with scipy
# Can be added back if needed with updated package versions
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (16, 10)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print("Imports complete!")

In [ ]:
# Configuration
import sys
sys.path.append(str(Path('../..')))

from config import (
    PROCESSED_WEEKLY_DIR,
    MASK_FILE,
    CATTLE_DATA_FILE,
    FIGURES_DIR,
    CATTLE_REGION_IDS
)

OUTPUT_DIR = FIGURES_DIR / 'vpd_mechanisms'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Load integrated dataset from previous notebook
# (Or recreate it here)
print("Loading data...")

# Load weekly climate data
ds_night = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_nighttime_recovery.nc')
ds_day = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_daytime_heat.nc')
ds_vpd = xr.open_dataset(PROCESSED_WEEKLY_DIR / 'weekly_vpd.nc')

week_dates = ds_vpd['week'].values

# Load mask
ds_mask = xr.open_dataset(MASK_FILE)
state_mask = ds_mask.state_mask.load()
ds_mask.close()

# Helper function
def compute_regional_mean(data, state_ids, state_mask):
    combined_mask = xr.zeros_like(state_mask, dtype=bool)
    for state_id in state_ids:
        combined_mask = combined_mask | (state_mask == state_id)
    masked_data = data.where(combined_mask)
    return masked_data.mean(dim=['lat', 'lon']).astype(np.float64)

# Compute regional metrics
region_4_ids = CATTLE_REGION_IDS['region_4']
region_6_ids = CATTLE_REGION_IDS['region_6']
combined_ids = region_4_ids + region_6_ids

# Extract metrics
df_climate = pd.DataFrame({
    'date': pd.to_datetime(week_dates),
    'vpd_mean': compute_regional_mean(ds_vpd['vpd_mean'], combined_ids, state_mask).values,
    'vpd_max': compute_regional_mean(ds_vpd['vpd_max'], combined_ids, state_mask).values,
    'temp_day': compute_regional_mean(ds_day['hours_above_30'], combined_ids, state_mask).values,
    'temp_night': compute_regional_mean(ds_night['hours_above_24'], combined_ids, state_mask).values,
    'temp_extreme': compute_regional_mean(ds_day['hours_above_35'], combined_ids, state_mask).values,
})

# Load cattle data
df_cattle = pd.read_csv(CATTLE_DATA_FILE, parse_dates=['date'])

# Merge
df_merged = pd.merge(df_cattle, df_climate, on='date', how='inner')
df_merged = df_merged.dropna()

# Add derived variables
df_merged['total_slaughter'] = df_merged['region_4_beef_dairy'] + df_merged['region_6_beef_dairy']
df_merged['year'] = df_merged['date'].dt.year
df_merged['month'] = df_merged['date'].dt.month
df_merged['season'] = df_merged['month'].apply(
    lambda m: 'Winter' if m in [12, 1, 2] else 
              'Spring' if m in [3, 4, 5] else 
              'Summer' if m in [6, 7, 8] else 'Fall'
)

print(f"Merged dataset: {df_merged.shape}")
print(f"Date range: {df_merged['date'].min()} to {df_merged['date'].max()}")

## 1. VPD Threshold Identification

Identify critical VPD levels where mortality risk increases.

In [ ]:
# Bin VPD and compute mean mortality
vpd_bins = np.arange(0, 5.5, 0.25)  # 0-5.5 kPa in 0.25 kPa bins
df_merged['vpd_bin'] = pd.cut(df_merged['vpd_mean'], bins=vpd_bins)

# Compute statistics by bin
binned_stats = df_merged.groupby('vpd_bin', observed=True).agg({
    'total_slaughter': ['mean', 'std', 'count'],
    'vpd_mean': 'mean'
}).reset_index()

binned_stats.columns = ['vpd_bin', 'slaughter_mean', 'slaughter_std', 'count', 'vpd_center']
binned_stats = binned_stats[binned_stats['count'] >= 10]  # At least 10 observations
binned_stats['slaughter_se'] = binned_stats['slaughter_std'] / np.sqrt(binned_stats['count'])

# Plot dose-response curve
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Plot 1: Mean response with error bars
ax = axes[0, 0]
ax.errorbar(binned_stats['vpd_center'], binned_stats['slaughter_mean'],
           yerr=binned_stats['slaughter_se'], fmt='o', capsize=5,
           markersize=8, color='darkblue', ecolor='gray', alpha=0.7)

# Add smooth curve (loess-like)
from scipy.interpolate import UnivariateSpline
if len(binned_stats) > 5:
    spl = UnivariateSpline(binned_stats['vpd_center'], binned_stats['slaughter_mean'], 
                          s=len(binned_stats)*100, k=3)
    x_smooth = np.linspace(binned_stats['vpd_center'].min(), 
                          binned_stats['vpd_center'].max(), 100)
    y_smooth = spl(x_smooth)
    ax.plot(x_smooth, y_smooth, 'r-', linewidth=2, alpha=0.7, label='Smoothed trend')

ax.set_xlabel('Mean VPD (kPa)', fontsize=12)
ax.set_ylabel('Mean Weekly Slaughter (1000 head)', fontsize=12)
ax.set_title('Dose-Response: VPD vs Cattle Slaughter\n(Combined Regions 4 & 6)',
            fontweight='bold', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Identify potential threshold using segmented regression
ax = axes[0, 1]

# Test different potential thresholds
threshold_candidates = np.linspace(1.5, 3.5, 20)
r2_scores = []

for threshold in threshold_candidates:
    # Create binary indicator
    below = df_merged['vpd_mean'] < threshold
    
    # Fit separate regressions
    X_below = df_merged.loc[below, 'vpd_mean'].values.reshape(-1, 1)
    y_below = df_merged.loc[below, 'total_slaughter'].values
    
    X_above = df_merged.loc[~below, 'vpd_mean'].values.reshape(-1, 1)
    y_above = df_merged.loc[~below, 'total_slaughter'].values
    
    if len(X_below) > 10 and len(X_above) > 10:
        # Fit models
        model_below = LinearRegression().fit(X_below, y_below)
        model_above = LinearRegression().fit(X_above, y_above)
        
        # Predict full dataset
        y_pred = np.where(below,
                         model_below.predict(df_merged['vpd_mean'].values.reshape(-1, 1)),
                         model_above.predict(df_merged['vpd_mean'].values.reshape(-1, 1)))
        
        r2 = r2_score(df_merged['total_slaughter'], y_pred)
        r2_scores.append(r2)
    else:
        r2_scores.append(np.nan)

# Plot R² vs threshold
ax.plot(threshold_candidates, r2_scores, 'o-', linewidth=2, markersize=6, color='darkgreen')
ax.set_xlabel('VPD Threshold (kPa)', fontsize=12)
ax.set_ylabel('R² (Segmented Model)', fontsize=12)
ax.set_title('Threshold Identification\nOptimal Breakpoint Analysis',
            fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

# Mark optimal threshold
if not all(np.isnan(r2_scores)):
    optimal_idx = np.nanargmax(r2_scores)
    optimal_threshold = threshold_candidates[optimal_idx]
    ax.axvline(optimal_threshold, color='red', linestyle='--', linewidth=2,
              label=f'Optimal: {optimal_threshold:.2f} kPa')
    ax.legend()

# Plot 3: Distribution of mortality by VPD category
ax = axes[1, 0]

# Define VPD categories
df_merged['vpd_category'] = pd.cut(df_merged['vpd_mean'], 
                                   bins=[0, 1.5, 2.5, 3.5, 6],
                                   labels=['Low (<1.5)', 'Moderate (1.5-2.5)', 
                                          'High (2.5-3.5)', 'Extreme (>3.5)'])

# Box plot
categories = ['Low (<1.5)', 'Moderate (1.5-2.5)', 'High (2.5-3.5)', 'Extreme (>3.5)']
data_by_cat = [df_merged[df_merged['vpd_category'] == cat]['total_slaughter'].dropna() 
               for cat in categories]

bp = ax.boxplot(data_by_cat, labels=categories, patch_artist=True,
               showmeans=True, meanline=True)

colors = ['lightblue', 'yellow', 'orange', 'red']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)

ax.set_xlabel('VPD Category', fontsize=12)
ax.set_ylabel('Weekly Slaughter (1000 head)', fontsize=12)
ax.set_title('Mortality Distribution by VPD Category',
            fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')

# Add sample sizes and means
for i, (cat, data) in enumerate(zip(categories, data_by_cat)):
    if len(data) > 0:
        ax.text(i+1, ax.get_ylim()[1]*0.98, f'n={len(data)}\nμ={data.mean():.0f}',
               ha='center', va='top', fontsize=8)

# Plot 4: Percentile response curves
ax = axes[1, 1]

percentiles = [10, 25, 50, 75, 90]
for p in percentiles:
    percentile_vals = binned_stats.groupby('vpd_center', observed=False)['slaughter_mean'].quantile(p/100)
    # Use running quantile from original data
    vpd_sorted = df_merged.sort_values('vpd_mean')
    window = len(vpd_sorted) // 20  # Rolling window
    rolling_p = vpd_sorted['total_slaughter'].rolling(window=window, center=True).quantile(p/100)
    
    ax.plot(vpd_sorted['vpd_mean'], rolling_p, linewidth=2, alpha=0.7, label=f'P{p}')

ax.set_xlabel('Mean VPD (kPa)', fontsize=12)
ax.set_ylabel('Weekly Slaughter (1000 head)', fontsize=12)
ax.set_title('Quantile Response Curves\nMortality Percentiles vs VPD',
            fontweight='bold', fontsize=13)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '01_vpd_threshold_identification.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\n=== Threshold Analysis Summary ===")
if not all(np.isnan(r2_scores)):
    print(f"Optimal VPD threshold: {optimal_threshold:.2f} kPa")
    print(f"R² at optimal threshold: {r2_scores[optimal_idx]:.3f}")

print(f"\nMean slaughter by VPD category:")
for cat in categories:
    data = df_merged[df_merged['vpd_category'] == cat]['total_slaughter']
    print(f"  {cat}: {data.mean():.1f} ± {data.std():.1f} (n={len(data)})")

## 2. Compound Stress Analysis: VPD × Temperature Interactions

Examine how VPD and temperature combine to affect mortality.

In [ ]:
# Create compound stress metrics
df_merged['vpd_temp_product'] = df_merged['vpd_mean'] * df_merged['temp_day']
df_merged['compound_index'] = (df_merged['vpd_mean'] / 3.0) * (df_merged['temp_day'] / 50)  # Normalized

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Plot 1: 3D surface plot (using binned statistics)
ax = axes[0, 0]

# Use binned statistics (more robust than interpolation)
from scipy.stats import binned_statistic_2d

# Define bins
n_bins = 25
vpd_bins = np.linspace(df_merged['vpd_mean'].min(), df_merged['vpd_mean'].max(), n_bins)
temp_bins = np.linspace(df_merged['temp_day'].min(), df_merged['temp_day'].max(), n_bins)

# Compute 2D binned mean
mortality_binned, vpd_edges, temp_edges, _ = binned_statistic_2d(
    df_merged['vpd_mean'].values,
    df_merged['temp_day'].values,
    df_merged['total_slaughter'].values,
    statistic='mean',
    bins=[vpd_bins, temp_bins]
)

# Create mesh grid for plotting (use bin centers)
vpd_centers = (vpd_edges[:-1] + vpd_edges[1:]) / 2
temp_centers = (temp_edges[:-1] + temp_edges[1:]) / 2
VPD, TEMP = np.meshgrid(vpd_centers, temp_centers)

# Transpose to match meshgrid orientation
MORTALITY = mortality_binned.T

# Contour plot
contour = ax.contourf(VPD, TEMP, MORTALITY, levels=15, cmap='YlOrRd')
plt.colorbar(contour, ax=ax, label='Slaughter (1000 head/week)')
ax.set_xlabel('Mean VPD (kPa)', fontsize=12)
ax.set_ylabel('Daytime Heat Hours >30°C (hrs/week)', fontsize=12)
ax.set_title('Compound Stress Response Surface\nMortality as Function of VPD and Temperature',
            fontweight='bold', fontsize=13)

# Plot 2: Interaction effect from regression
ax = axes[0, 1]

# Fit model with interaction
X = df_merged[['vpd_mean', 'temp_day']].values
X_interaction = np.column_stack([X, X[:, 0] * X[:, 1]])  # Add interaction term
X_with_const = add_constant(X_interaction)
y = df_merged['total_slaughter'].values

model_interaction = OLS(y, X_with_const).fit()

# Plot interaction effects at different VPD levels
vpd_levels = [1.0, 2.0, 3.0, 4.0]
temp_range = np.linspace(0, 80, 100)

for vpd_level in vpd_levels:
    # Create prediction matrix matching the training data structure
    # Columns: [vpd_mean, temp_day, vpd*temp]
    vpd_col = np.full(100, vpd_level, dtype=np.float64)
    temp_col = temp_range.astype(np.float64)
    interaction_col = (vpd_level * temp_range).astype(np.float64)
    
    X_pred = np.column_stack([vpd_col, temp_col, interaction_col])
    
    # Add constant term (will be added as first column by default)
    X_pred_const = add_constant(X_pred, has_constant='add')
    
    # Ensure shapes match
    if X_pred_const.shape[1] != model_interaction.params.shape[0]:
        raise ValueError(f"Shape mismatch: X_pred has {X_pred_const.shape[1]} columns but model expects {model_interaction.params.shape[0]}")
    
    y_pred = model_interaction.predict(X_pred_const)
    
    ax.plot(temp_range, y_pred, linewidth=2, label=f'VPD={vpd_level:.1f} kPa')

ax.set_xlabel('Daytime Heat Hours >30°C (hrs/week)', fontsize=12)
ax.set_ylabel('Predicted Slaughter (1000 head/week)', fontsize=12)
ax.set_title('VPD × Temperature Interaction\nTemperature Effect at Different VPD Levels',
            fontweight='bold', fontsize=13)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

# Plot 3: Compound index relationship
ax = axes[1, 0]

# Bin compound index
comp_bins = np.percentile(df_merged['compound_index'], [0, 25, 50, 75, 100])
df_merged['comp_bin'] = pd.cut(df_merged['compound_index'], bins=comp_bins,
                                labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'],
                                include_lowest=True)

# Box plot
data_by_comp = [df_merged[df_merged['comp_bin'] == label]['total_slaughter'].dropna()
                for label in ['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)']]

bp = ax.boxplot(data_by_comp, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'],
               patch_artist=True, showmeans=True, meanline=True)

for patch, color in zip(bp['boxes'], plt.cm.Reds(np.linspace(0.3, 0.9, 4))):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xlabel('Compound Stress Quartile', fontsize=12)
ax.set_ylabel('Weekly Slaughter (1000 head)', fontsize=12)
ax.set_title('Mortality by Compound Stress Index\n(VPD × Temperature)',
            fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')

# Add statistics
for i, data in enumerate(data_by_comp):
    ax.text(i+1, ax.get_ylim()[1]*0.98, f'μ={data.mean():.0f}\nn={len(data)}',
           ha='center', va='top', fontsize=9)

# Plot 4: Synergy analysis
ax = axes[1, 1]

# Define high/low categories
vpd_high = df_merged['vpd_mean'] > df_merged['vpd_mean'].median()
temp_high = df_merged['temp_day'] > df_merged['temp_day'].median()

# Four categories
categories_stress = [
    ('Low VPD\nLow Temp', ~vpd_high & ~temp_high),
    ('Low VPD\nHigh Temp', ~vpd_high & temp_high),
    ('High VPD\nLow Temp', vpd_high & ~temp_high),
    ('High VPD\nHigh Temp', vpd_high & temp_high),
]

means = [df_merged[mask]['total_slaughter'].mean() for _, mask in categories_stress]
sems = [df_merged[mask]['total_slaughter'].sem() for _, mask in categories_stress]
labels = [label for label, _ in categories_stress]

bars = ax.bar(range(4), means, yerr=sems, capsize=5, alpha=0.7,
             color=['lightblue', 'orange', 'lightcoral', 'darkred'],
             edgecolor='black')

ax.set_xticks(range(4))
ax.set_xticklabels(labels)
ax.set_ylabel('Mean Slaughter (1000 head/week)', fontsize=12)
ax.set_title('Synergistic Effects of VPD and Temperature\nFour-Category Comparison',
            fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')

# Test for synergy (interaction beyond additive)
additive_expected = (means[1] - means[0]) + (means[2] - means[0]) + means[0]
observed = means[3]
synergy = observed - additive_expected

ax.text(0.02, 0.98, f'Expected (additive): {additive_expected:.1f}\nObserved: {observed:.1f}\nSynergy: {synergy:.1f}',
       transform=ax.transAxes, va='top',
       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8), fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '02_compound_stress_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== Compound Stress Model Results ===")
print(model_interaction.summary())
print(f"\n=== Synergy Analysis ===")
print(f"Additive expectation: {additive_expected:.1f}")
print(f"Observed (high VPD + high temp): {observed:.1f}")
print(f"Synergistic effect: {synergy:.1f} ({synergy/additive_expected*100:.1f}% above additive)")

## 3. Cumulative VPD Exposure Analysis

Assess effects of sustained vs. acute VPD stress.

In [ ]:
# Compute cumulative exposure metrics
windows = [2, 4, 8]  # weeks

for window in windows:
    df_merged[f'vpd_mean_{window}wk'] = df_merged['vpd_mean'].rolling(window=window).mean()
    df_merged[f'vpd_max_{window}wk'] = df_merged['vpd_max'].rolling(window=window).mean()
    df_merged[f'vpd_cumsum_{window}wk'] = df_merged['vpd_mean'].rolling(window=window).sum()

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Plot 1: Acute vs cumulative
ax = axes[0, 0]

# Scatter: current week vs 4-week average
mask_complete = df_merged['vpd_mean_4wk'].notna() & df_merged['total_slaughter'].notna()
x_acute = df_merged.loc[mask_complete, 'vpd_mean'].values
x_cumulative = df_merged.loc[mask_complete, 'vpd_mean_4wk'].values
y = df_merged.loc[mask_complete, 'total_slaughter'].values

# Compare correlations
r_acute, p_acute = stats.pearsonr(x_acute, y)
r_cumulative, p_cumulative = stats.pearsonr(x_cumulative, y)

# Plot both
ax.scatter(x_acute, y, alpha=0.3, s=20, color='blue', label=f'Acute (r={r_acute:.3f})')
ax.scatter(x_cumulative, y, alpha=0.3, s=20, color='red', label=f'4-week avg (r={r_cumulative:.3f})')

# Regression lines
z1 = np.polyfit(x_acute, y, 1)
z2 = np.polyfit(x_cumulative, y, 1)
x_line = np.linspace(min(x_acute.min(), x_cumulative.min()),
                    max(x_acute.max(), x_cumulative.max()), 100)
ax.plot(x_line, np.poly1d(z1)(x_line), 'b--', linewidth=2, alpha=0.7)
ax.plot(x_line, np.poly1d(z2)(x_line), 'r--', linewidth=2, alpha=0.7)

ax.set_xlabel('Mean VPD (kPa)', fontsize=12)
ax.set_ylabel('Weekly Slaughter (1000 head)', fontsize=12)
ax.set_title('Acute vs Cumulative VPD Exposure\nCurrent Week vs 4-Week Average',
            fontweight='bold', fontsize=13)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)

# Plot 2: Correlation by window size
ax = axes[0, 1]

window_sizes = list(range(1, 13))  # 1-12 weeks
correlations_window = []

for w in window_sizes:
    rolling_vpd = df_merged['vpd_mean'].rolling(window=w).mean()
    mask = rolling_vpd.notna() & df_merged['total_slaughter'].notna()
    if mask.sum() > 10:
        r, _ = stats.pearsonr(rolling_vpd[mask], df_merged.loc[mask, 'total_slaughter'])
        correlations_window.append(r)
    else:
        correlations_window.append(np.nan)

ax.plot(window_sizes, correlations_window, 'o-', linewidth=2, markersize=8, color='darkgreen')
ax.axhline(0, color='black', linewidth=1)
ax.set_xlabel('Averaging Window (weeks)', fontsize=12)
ax.set_ylabel('Pearson Correlation', fontsize=12)
ax.set_title('Optimal Exposure Window\nCorrelation vs Averaging Period',
            fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

# Mark optimal window
if not all(np.isnan(correlations_window)):
    optimal_window = window_sizes[np.nanargmax(np.abs(correlations_window))]
    ax.axvline(optimal_window, color='red', linestyle='--', linewidth=2,
              label=f'Optimal: {optimal_window} weeks')
    ax.legend()

# Plot 3: Threshold exceedance duration
ax = axes[1, 0]

# Count consecutive weeks above threshold
threshold_vpd = 2.5  # kPa
above_threshold = (df_merged['vpd_mean'] > threshold_vpd).astype(int)

# Find consecutive streaks
streak_lengths = []
current_streak = 0

for val in above_threshold:
    if val == 1:
        current_streak += 1
    else:
        if current_streak > 0:
            streak_lengths.append(current_streak)
        current_streak = 0

if current_streak > 0:
    streak_lengths.append(current_streak)

# Histogram of streak lengths
if len(streak_lengths) > 0:
    ax.hist(streak_lengths, bins=range(1, max(streak_lengths)+2),
           color='coral', alpha=0.7, edgecolor='black')
    
    ax.set_xlabel('Consecutive Weeks Above Threshold', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title(f'Duration of High VPD Events\n(>{threshold_vpd} kPa sustained exposure)',
                fontweight='bold', fontsize=13)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add statistics
    stats_text = f'Total events: {len(streak_lengths)}\n'
    stats_text += f'Mean duration: {np.mean(streak_lengths):.1f} weeks\n'
    stats_text += f'Max duration: {max(streak_lengths)} weeks'
    ax.text(0.98, 0.98, stats_text, transform=ax.transAxes,
           ha='right', va='top',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Plot 4: Time-weighted exposure model
ax = axes[1, 1]

# Create decay-weighted exposure
decay_weights = [1.0, 0.8, 0.6, 0.4]  # Current week has weight 1.0, previous weeks decay
weighted_exposure = np.zeros(len(df_merged))

for i in range(len(df_merged)):
    for j, weight in enumerate(decay_weights):
        if i - j >= 0:
            weighted_exposure[i] += df_merged.iloc[i-j]['vpd_mean'] * weight

df_merged['vpd_weighted'] = weighted_exposure

# Compare weighted vs simple average
mask_w = df_merged['vpd_weighted'].notna() & df_merged['total_slaughter'].notna()
r_weighted, _ = stats.pearsonr(df_merged.loc[mask_w, 'vpd_weighted'],
                               df_merged.loc[mask_w, 'total_slaughter'])

# Scatter plot
ax.scatter(df_merged['vpd_weighted'], df_merged['total_slaughter'],
          alpha=0.4, s=20, color='purple')

# Regression
x_w = df_merged.loc[mask_w, 'vpd_weighted'].values
y_w = df_merged.loc[mask_w, 'total_slaughter'].values
z = np.polyfit(x_w, y_w, 1)
x_line = np.linspace(x_w.min(), x_w.max(), 100)
ax.plot(x_line, np.poly1d(z)(x_line), 'k--', linewidth=2)

ax.set_xlabel('Time-Weighted VPD Exposure', fontsize=12)
ax.set_ylabel('Weekly Slaughter (1000 head)', fontsize=12)
ax.set_title(f'Decay-Weighted Exposure Model\n(r = {r_weighted:.3f})',
            fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

ax.text(0.05, 0.95, f'Weights: {decay_weights}\n(current → past)',
       transform=ax.transAxes, va='top',
       bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

plt.tight_layout()
plt.savefig(OUTPUT_DIR / '03_cumulative_exposure_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n=== Cumulative Exposure Summary ===")
print(f"Acute (current week) correlation: r = {r_acute:.3f}")
print(f"4-week average correlation: r = {r_cumulative:.3f}")
print(f"Weighted exposure correlation: r = {r_weighted:.3f}")
if not all(np.isnan(correlations_window)):
    print(f"Optimal averaging window: {optimal_window} weeks")
print(f"\nHigh VPD events (>{threshold_vpd} kPa):")
if len(streak_lengths) > 0:
    print(f"  Total events: {len(streak_lengths)}")
    print(f"  Mean duration: {np.mean(streak_lengths):.1f} weeks")
    print(f"  Max duration: {max(streak_lengths)} weeks")